# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

## 1. Method choice and why

I chose Logistic Regression because the goal is to predict whether a page represents a directional CTR opportunity.

Logistic Regression is appropriate because the target can be represented as a binary outcome and the model is simple and interpretable. It also provides a useful comparison against the Week-4 rule-based baseline without rewarding complexity alone.

The main features are decision-time page-level signals such as impressions, clicks, CTR, average position, and sessions. I will not use future outcome fields or fields derived from the target.

The purpose of this model is decision support, not causal inference. A positive prediction means that the page resembles pages identified as opportunities by the defined proxy label; it does not prove that changing the page will improve CTR.

In [ ]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{DATA_PATH}')
    WHERE month = '2026-03'
""").df()

print("Connected to Hugging Face warehouse.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

## 2. Split design

I will use a grouped split by pseudonymized client so that pages from the same client do not appear in both training and test data.

This is an honest split because pages from the same client can have similar characteristics. Keeping clients separated reduces the risk that the model appears stronger simply because it has already seen data from the same client.

I will use the same development month, March 2026, and will keep the final `_sample` month sealed.

The test set will be held out before model training and the baseline and Logistic Regression model will be evaluated on the same test rows.

In [ ]:
# Check that the client identifier needed for grouped splitting exists

if "client_hash_id" not in df.columns:
    raise ValueError(
        "client_hash_id was not found. "
        "Check the available columns printed in Section 1."
    )

print("Client identifier found:", "client_hash_id")
print("Unique clients:", df["client_hash_id"].nunique())

## 3. Train + compare vs my baseline

## 3. Train + compare vs my baseline

I will build a simple binary opportunity label using the CTR-versus-position idea from the earlier baseline work.

A page is treated as a directional opportunity when it has enough impressions and its observed CTR is below the typical CTR for its position tier.

This is a proxy label, not a future business outcome.

I will train Logistic Regression using decision-time signals only and compare it with the Week-4 baseline on the same held-out rows.

The main comparison metric is ROC-AUC, with average precision also reported because the opportunity classes may not be balanced.

In [ ]:
# Check required columns

required_columns = [
    "impressions",
    "clicks",
    "ctr",
    "average_position",
    "client_hash_id"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

model_df = df.copy()

# Convert numeric columns
numeric_columns = [
    "impressions",
    "clicks",
    "ctr",
    "average_position"
]

for col in numeric_columns:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Keep rows with usable search exposure
model_df = model_df[
    model_df["impressions"].notna()
    & (model_df["impressions"] >= 100)
    & model_df["ctr"].notna()
    & model_df["average_position"].notna()
].copy()

print("Rows after filtering:", len(model_df))

In [ ]:
# Create position tiers

model_df["position_tier"] = pd.cut(
    model_df["average_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"]
)

# Calculate typical CTR for each position tier
expected_ctr = (
    model_df
    .groupby("position_tier", observed=True)["ctr"]
    .median()
)

model_df["expected_ctr"] = (
    model_df["position_tier"]
    .map(expected_ctr)
    .astype(float)
)

# Directional opportunity label
model_df["opportunity_label"] = (
    model_df["ctr"] < model_df["expected_ctr"]
).astype(int)

print("Label counts:")
print(model_df["opportunity_label"].value_counts())

print("\nLabel proportions:")
print(model_df["opportunity_label"].value_counts(normalize=True))

In [ ]:
# Create a client-level train/test split

clients = model_df["client_hash_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_hash_id"].isin(test_clients)
].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("Training clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

print(
    "\nClient overlap:",
    len(
        set(train_df["client_hash_id"])
        & set(test_df["client_hash_id"])
    )
)

In [ ]:
# Select decision-time features only

candidate_features = [
    "impressions",
    "clicks",
    "ctr",
    "average_position"
]

if "sessions" in model_df.columns:
    candidate_features.append("sessions")

if "sessions_paid" in model_df.columns:
    candidate_features.append("sessions_paid")

features = [
    col for col in candidate_features
    if col in model_df.columns
]

print("Features used:")
print(features)

X_train = train_df[features]
y_train = train_df["opportunity_label"]

X_test = test_df[features]
y_test = test_df["opportunity_label"]

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")

In [ ]:
# Week-4 directional baseline:
# Larger CTR gap = stronger opportunity signal

baseline_score = (
    test_df["expected_ctr"] - test_df["ctr"]
)

baseline_prediction = (
    baseline_score > 0
).astype(int)

model_prediction = (
    model_probability >= 0.5
).astype(int)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "ROC_AUC": [
        roc_auc_score(
            y_test,
            baseline_score
        ),
        roc_auc_score(
            y_test,
            model_probability
        )
    ],
    "Average_Precision": [
        average_precision_score(
            y_test,
            baseline_score
        ),
        average_precision_score(
            y_test,
            model_probability
        )
    ],
    "Precision": [
        precision_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        precision_score(
            y_test,
            model_prediction,
            zero_division=0
        )
    ],
    "Recall": [
        recall_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        recall_score(
            y_test,
            model_prediction,
            zero_division=0
        )
    ]
})

display(comparison)

## 4. Errors and interpretation

I will inspect false positives and false negatives rather than relying only on the overall metrics.

A false positive is a page that the model predicts as an opportunity but does not match the proxy label.

A false negative is a page that matches the proxy label but receives a low predicted opportunity probability.

These errors matter because the label itself is only a directional CTR opportunity proxy. The model cannot observe every factor that affects CTR, such as query intent, SERP features, brand effects, or differences between clients.

The model coefficients will also be inspected to understand which signals the model relies on most strongly.

The results should be interpreted as decision-support signals, not causal evidence that changing a page will improve performance.

In [ ]:
# Build an error-analysis table

errors = test_df[
    [
        "content_hash_id",
        "report_date",
        "ctr",
        "expected_ctr",
        "average_position"
    ]
].copy()

errors["actual"] = y_test.values
errors["predicted"] = model_prediction
errors["probability"] = model_probability

errors["error_type"] = np.select(
    [
        (errors["actual"] == 0) &
        (errors["predicted"] == 1),

        (errors["actual"] == 1) &
        (errors["predicted"] == 0)
    ],
    [
        "false_positive",
        "false_negative"
    ],
    default="correct"
)

print("Error counts:")
print(errors["error_type"].value_counts())

display(
    errors[
        errors["error_type"] != "correct"
    ].head(20)
)

In [ ]:
# Inspect Logistic Regression coefficients

classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": classifier.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coefficients)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.